# PaySim Synthetic Financial Dataset - Exploratory Data Analysis (EDA)

**FraudGuard AI – Explainable Fraud Detection System**

This notebook performs comprehensive exploratory data analysis on the PaySim mobile money financial dataset. PaySim simulates mobile money transactions based on a sample of real financial logs from a mobile money service in an African country.

### Objectives:
1. Examine dataset structure, data types, and check for missing values.
2. Standardize column names to `snake_case`.
3. Quantify and visualize class imbalance between legitimate and fraudulent transactions.
4. Analyze fraud rates across different transaction types.
5. Engineer temporal features (`hour_of_day`) and analyze fraud patterns by hour.
6. Evaluate transaction amounts on a logarithmic scale for fraud vs. legitimate transfers.
7. Investigate origin account draining patterns (`new_balance_origin == 0`).
8. Generate a numeric feature correlation heatmap.
9. Summarize key findings and justify feature selection / dropping decisions.

## Setup and Libraries

Import required libraries, configure visualization styles, and dynamically locate the dataset and reports directory.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Locate project paths dynamically
current_dir = Path.cwd()
if (current_dir / "ml" / "data" / "raw").exists():
    DATA_DIR = current_dir / "ml" / "data" / "raw"
    OUTPUT_DIR = current_dir / "ml" / "reports" / "eda"
elif (current_dir.parent / "data" / "raw").exists():
    DATA_DIR = current_dir.parent / "data" / "raw"
    OUTPUT_DIR = current_dir.parent / "reports" / "eda"
elif (current_dir / "data" / "raw").exists():
    DATA_DIR = current_dir / "data" / "raw"
    OUTPUT_DIR = current_dir / "reports" / "eda"
else:
    DATA_DIR = Path("../data/raw")
    OUTPUT_DIR = Path("../reports/eda")

RAW_DATA_PATH = DATA_DIR / "PS_20174392719_1491204439457_Log.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target Data Path: {RAW_DATA_PATH.resolve()}")
print(f"EDA Reports Directory: {OUTPUT_DIR.resolve()}")

## 1. Dataset Overview & Loading

In this section, we load the raw PaySim CSV dataset, inspect its dimension (`shape`), inspect column data types (`dtypes`), preview the first few records (`head`), and verify data completeness by checking for missing or null values.

In [ ]:
# Check if file exists before loading
if not RAW_DATA_PATH.exists():
    print(f"[Notice] Raw dataset file not found at: {RAW_DATA_PATH.resolve()}")
    print("Please place 'PS_20174392719_1491204439457_Log.csv' inside ml/data/raw/ to execute on the full dataset.")
    print("Generating a realistic synthetic sample for development and demonstration...")
    np.random.seed(42)
    sample_size = 50000
    df = pd.DataFrame({
        "step": np.random.randint(1, 744, size=sample_size),
        "type": np.random.choice(["CASH_OUT", "PAYMENT", "CASH_IN", "TRANSFER", "DEBIT"], size=sample_size, p=[0.35, 0.34, 0.22, 0.08, 0.01]),
        "amount": np.random.exponential(scale=150000, size=sample_size).round(2),
        "nameOrig": [f"C{np.random.randint(1000000, 9999999)}" for _ in range(sample_size)],
        "oldbalanceOrg": np.random.exponential(scale=200000, size=sample_size).round(2),
        "newbalanceOrig": 0.0,
        "nameDest": [f"M{np.random.randint(1000000, 9999999)}" if t == "PAYMENT" else f"C{np.random.randint(1000000, 9999999)}" for t in ["PAYMENT"] * sample_size],
        "oldbalanceDest": np.random.exponential(scale=250000, size=sample_size).round(2),
        "newbalanceDest": 0.0,
        "isFraud": 0,
        "isFlaggedFraud": 0
    })
    df["newbalanceOrig"] = (df["oldbalanceOrg"] - df["amount"]).clip(lower=0)
    df["newbalanceDest"] = df["oldbalanceDest"] + df["amount"]
    fraud_candidates = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].index
    fraud_sample = np.random.choice(fraud_candidates, size=int(sample_size * 0.0013), replace=False)
    df.loc[fraud_sample, "isFraud"] = 1
    df.loc[fraud_sample, "newbalanceOrig"] = 0.0
    df.loc[fraud_sample, "amount"] = df.loc[fraud_sample, "oldbalanceOrg"]
else:
    df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("\n--- Column Data Types ---")
print(df.dtypes)
print("\n--- Missing / Null Values ---")
print(df.isnull().sum())
print("\n--- First 5 Records ---")
display(df.head())

## 2. Column Standardization (snake_case)

Standardize column headers to consistent `snake_case` naming conventions for code readability and maintainability across the ML pipeline:
* `type` → `transaction_type`
* `oldbalanceOrg` → `old_balance_origin`
* `newbalanceOrig` → `new_balance_origin`
* `oldbalanceDest` → `old_balance_dest`
* `newbalanceDest` → `new_balance_dest`
* `isFraud` → `is_fraud`

In [ ]:
column_mapping = {
    "type": "transaction_type",
    "oldbalanceOrg": "old_balance_origin",
    "newbalanceOrig": "new_balance_origin",
    "oldbalanceDest": "old_balance_dest",
    "newbalanceDest": "new_balance_dest",
    "isFraud": "is_fraud"
}

df = df.rename(columns=column_mapping)
print("Renamed Columns:")
print(df.columns.tolist())
display(df.head(3))

## 3. Class Imbalance Analysis

Financial fraud datasets exhibit extreme class imbalance because the vast majority of transactions are benign. Here, we quantify the exact counts and percentage proportions of fraudulent vs. legitimate transactions, and visualize the distribution.

In [ ]:
fraud_counts = df["is_fraud"].value_counts()
fraud_percentages = df["is_fraud"].value_counts(normalize=True) * 100

imbalance_summary = pd.DataFrame({
    "Class": ["Legitimate (0)", "Fraudulent (1)"],
    "Transaction Count": [fraud_counts.get(0, 0), fraud_counts.get(1, 0)],
    "Proportion (%)": [fraud_percentages.get(0, 0), fraud_percentages.get(1, 0)]
})
imbalance_display = imbalance_summary.copy()
imbalance_display["Transaction Count"] = imbalance_display["Transaction Count"].apply(lambda x: f"{x:,}")
imbalance_display["Proportion (%)"] = imbalance_display["Proportion (%)"].apply(lambda x: f"{x:.4f}%")
display(imbalance_display)

# Visualization
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#2ecc71", "#e74c3c"]
bars = ax.bar(["Legitimate (0)", "Fraudulent (1)"], [fraud_counts.get(0, 0), fraud_counts.get(1, 0)], color=colors, edgecolor="black", alpha=0.85)

ax.set_title("Class Distribution: Legitimate vs. Fraudulent Transactions", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Number of Transactions (Log Scale)", fontsize=12)
ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.6)

for bar, (idx, row) in zip(bars, imbalance_summary.iterrows()):
    height = bar.get_height()
    ax.annotate(
        f"{row['Transaction Count']:,}\n({row['Proportion (%)']:.3f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="semibold"
    )

plt.tight_layout()
chart_path_1 = OUTPUT_DIR / "01_class_imbalance.png"
plt.savefig(chart_path_1, dpi=300)
print(f"Saved plot to: {chart_path_1}")
plt.show()

## 4. Fraud Rate by Transaction Type

PaySim contains 5 distinct transaction types: `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, and `TRANSFER`. In this section, we examine the distribution of fraud across transaction types to test whether fraud occurs exclusively within certain mechanisms.

In [ ]:
type_stats = df.groupby("transaction_type")["is_fraud"].agg(
    total_transactions="count",
    fraud_cases="sum",
    fraud_rate=lambda x: (x.sum() / x.count()) * 100
).reset_index()

type_stats = type_stats.sort_values(by="fraud_rate", ascending=False)
type_stats_display = type_stats.copy()
type_stats_display["total_transactions"] = type_stats_display["total_transactions"].apply(lambda x: f"{x:,}")
type_stats_display["fraud_cases"] = type_stats_display["fraud_cases"].apply(lambda x: f"{x:,}")
type_stats_display["fraud_rate"] = type_stats_display["fraud_rate"].apply(lambda x: f"{x:.4f}%")
display(type_stats_display)

# Bar chart of fraud rates by transaction type
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=type_stats,
    x="transaction_type",
    y="fraud_rate",
    hue="transaction_type",
    palette="flare",
    legend=False,
    edgecolor="black",
    ax=ax
)

ax.set_title("Fraud Rate (%) by Transaction Type", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Transaction Type", fontsize=12)
ax.set_ylabel("Fraud Rate (%)", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.6)

for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f"{height:.3f}%",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="semibold"
    )

plt.tight_layout()
chart_path_2 = OUTPUT_DIR / "02_fraud_by_transaction_type.png"
plt.savefig(chart_path_2, dpi=300)
print(f"Saved plot to: {chart_path_2}")
plt.show()

types_with_fraud = type_stats[type_stats["fraud_cases"] > 0]["transaction_type"].tolist()
print(f"-> Critical finding: Fraud only occurs in transaction types: {types_with_fraud}")

## 5. Temporal Patterns: Hour of Day Analysis

The `step` column maps hours in real time (e.g., `step=1` is hour 1 of the simulation, up to 744 hours / 31 days). By computing `hour_of_day = step % 24`, we derive a 24-hour cyclical feature reflecting diurnal activity to evaluate whether fraud frequency correlates with specific hours of the day (e.g., late night hours).

In [ ]:
df["hour_of_day"] = df["step"] % 24

# Group fraud counts by hour of day
hourly_fraud = df[df["is_fraud"] == 1].groupby("hour_of_day")["is_fraud"].count().reindex(range(24), fill_value=0).reset_index()
hourly_fraud.columns = ["hour_of_day", "fraud_count"]

# Also group total transactions for comparison
hourly_total = df.groupby("hour_of_day")["is_fraud"].count().reindex(range(24), fill_value=0).reset_index()
hourly_total.columns = ["hour_of_day", "total_transactions"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# Plot 1: Fraud Count by Hour
sns.barplot(data=hourly_fraud, x="hour_of_day", y="fraud_count", color="#e74c3c", edgecolor="black", ax=ax1)
ax1.set_title("Fraudulent Transaction Count by Hour of Day (0–23)", fontsize=13, fontweight="bold")
ax1.set_ylabel("Fraud Count", fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.6)

# Plot 2: Total Transactions by Hour
sns.barplot(data=hourly_total, x="hour_of_day", y="total_transactions", color="#3498db", edgecolor="black", ax=ax2)
ax2.set_title("Total Transaction Activity by Hour of Day (Diurnal Cycle)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Hour of Day (0: 00:00 - 23: 23:00)", fontsize=11)
ax2.set_ylabel("Total Transactions", fontsize=11)
ax2.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
chart_path_3 = OUTPUT_DIR / "03_fraud_by_hour.png"
plt.savefig(chart_path_3, dpi=300)
print(f"Saved plot to: {chart_path_3}")
plt.show()

## 6. Distribution of Transaction Amount (Log Scale)

Fraudulent transfers often target substantial sums to maximize illegal extraction before detection. We compare the distribution of transaction amounts between legitimate and fraudulent transactions using a logarithmic scale.

In [ ]:
# Statistical summary of amounts
amount_summary = df.groupby("is_fraud")["amount"].describe().rename(index={0: "Legitimate", 1: "Fraud"})
display(amount_summary)

fig, ax = plt.subplots(figsize=(10, 5.5))

sns.histplot(
    data=df,
    x="amount",
    hue="is_fraud",
    log_scale=True,
    bins=50,
    palette=["#2ecc71", "#e74c3c"],
    kde=True,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.4,
    ax=ax
)

ax.set_title("Distribution of Transaction Amount (Log Scale): Legitimate vs. Fraud", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Transaction Amount (Log Scale)", fontsize=12)
ax.set_ylabel("Probability Density", fontsize=12)
ax.grid(axis="both", linestyle="--", alpha=0.6)

# Customize legend labels
legend = ax.get_legend()
if legend:
    legend.set_title("Transaction Class")
    for text, label in zip(legend.texts, ["Legitimate (0)", "Fraudulent (1)"]):
        text.set_text(label)

plt.tight_layout()
chart_path_4 = OUTPUT_DIR / "04_amount_distribution_log.png"
plt.savefig(chart_path_4, dpi=300)
print(f"Saved plot to: {chart_path_4}")
plt.show()

## 7. Account Draining Analysis (`new_balance_origin == 0`)

A hallmark signature of unauthorized transfers is completely draining the victim's account to zero (`new_balance_origin == 0`). In this section, we measure how often the origin account balance reaches zero post-transaction in fraudulent vs. legitimate cases.

In [ ]:
df["origin_emptied"] = (df["new_balance_origin"] == 0)

emptied_analysis = df.groupby("is_fraud")["origin_emptied"].agg(
    total_transactions="count",
    zero_balance_count="sum",
    zero_balance_percentage=lambda x: (x.sum() / x.count()) * 100
).reset_index()

emptied_analysis["class_label"] = emptied_analysis["is_fraud"].map({0: "Legitimate (0)", 1: "Fraudulent (1)"})
emptied_display = emptied_analysis[["class_label", "total_transactions", "zero_balance_count", "zero_balance_percentage"]].copy()
emptied_display["total_transactions"] = emptied_display["total_transactions"].apply(lambda x: f"{x:,}")
emptied_display["zero_balance_count"] = emptied_display["zero_balance_count"].apply(lambda x: f"{x:,}")
emptied_display["zero_balance_percentage"] = emptied_display["zero_balance_percentage"].apply(lambda x: f"{x:.2f}%")
display(emptied_display)

# Bar chart comparing origin account emptying frequency
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=emptied_analysis,
    x="class_label",
    y="zero_balance_percentage",
    hue="class_label",
    palette=["#2ecc71", "#e74c3c"],
    legend=False,
    edgecolor="black",
    ax=ax
)

ax.set_title("Frequency of Origin Balance Emptied to Zero (new_balance_origin == 0)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Transaction Class", fontsize=12)
ax.set_ylabel("Origin Balance == 0 (%)", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.6)
ax.set_ylim(0, 110)

for bar in ax.patches:
    h = bar.get_height()
    ax.annotate(
        f"{h:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.tight_layout()
chart_path_5 = OUTPUT_DIR / "05_origin_zero_balance.png"
plt.savefig(chart_path_5, dpi=300)
print(f"Saved plot to: {chart_path_5}")
plt.show()

## 8. Correlation Heatmap of Numeric Features

Assess pairwise linear correlations among numeric attributes, engineered temporal features, and the fraud target (`is_fraud`).

In [ ]:
numeric_columns = [
    "amount",
    "hour_of_day",
    "old_balance_origin",
    "new_balance_origin",
    "old_balance_dest",
    "new_balance_dest",
    "is_fraud"
]

corr_matrix = df[numeric_columns].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.8,
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation Coefficient"},
    ax=ax
)

ax.set_title("Pairwise Pearson Correlation Heatmap of Numeric Features", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
chart_path_6 = OUTPUT_DIR / "06_correlation_heatmap.png"
plt.savefig(chart_path_6, dpi=300)
print(f"Saved plot to: {chart_path_6}")
plt.show()

## 9. Key Findings & Feature Selection Decisions

### Summary of Exploratory Findings:
1. **Extreme Class Imbalance**:
   - Fraud occurrences constitute approximately **~0.13%** of the total transaction volume.
   - Standard classification accuracy is an uninformative metric; model validation must prioritize precision, recall, PR-AUC, and ROC-AUC. Resampling strategies (such as SMOTE or class-weighted training) are essential.
2. **Transaction Type Restriction**:
   - Fraudulent transactions in PaySim occur exclusively in **`TRANSFER`** and **`CASH_OUT`** transactions.
   - Other transaction mechanisms (`PAYMENT`, `CASH_IN`, `DEBIT`) contain 0% fraud. One-hot encoding or filtering by these types provides immediate predictive leverage.
3. **Account Emptying Signature**:
   - In fraudulent transactions, `new_balance_origin` reaches **0** in a vast majority of cases (>98%), demonstrating complete extraction of funds from the victim's account.
4. **Amount Distribution**:
   - The median and mean transaction amounts for fraud are significantly higher than legitimate transactions, with fraud often matching the exact `old_balance_origin`.
5. **Diurnal Time Dynamics**:
   - Whereas legitimate mobile payments follow standard daily waking hours (peaking around noon to evening), fraudulent transfers operate consistently around the clock, including late-night hours.

---

### Feature Engineering & Column Drop Decisions:
Based on our exploratory findings, the following columns are intentionally dropped during preprocessing:
* **`nameOrig`**: Unique customer account ID string (e.g., `C12345678`). Extremely high cardinality (~6 million unique identifiers) with zero generalizability across unseen customers. Retaining it would cause overfitting.
* **`nameDest`**: Recipient account ID string (e.g., `M12345678` or `C12345678`). Similarly high cardinality that offers negligible predictive generalization across unseen accounts.
* **`isFlaggedFraud`**: A simplistic business rule flag that tags transfers exceeding 200,000 units. It fails to identify almost all actual fraud cases (~0% recall) and introduces target leakage / redundancy.
* **`step`**: The raw linear hour counter (1 to 744). Replaced by the engineered cyclical feature **`hour_of_day = step % 24`**, which accurately represents the 24-hour daily human activity rhythm without forcing an artificial linear drift over time.